# Phase 6.2 — SAP Evidence Retrieval
Uses the same verified body chunks from the 19-case Phase52 run. No DeepSeek. Local BGE embeddings will be calculated. Historical claim overlap is not gold recall.


In [ ]:
from pathlib import Path
import sys, subprocess, importlib, shutil, json
REPO_URL = 'https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git'
BRANCH = 'simplified-rag-v3'
ROOT = Path('/content/tfl-simplified-rag-v3')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{ROOT}[retrieval,test]'],check=True)
SRC = ROOT/'src'
sys.path.insert(0,str(SRC))
for name in list(sys.modules):
    if name=='tfl_agent' or name.startswith('tfl_agent.'):
        del sys.modules[name]
for p in SRC.joinpath('tfl_agent').rglob('__pycache__'):
    shutil.rmtree(p)
importlib.invalidate_caches()
import tfl_agent
assert Path(tfl_agent.__file__).resolve().is_relative_to(SRC.resolve())
print('Commit:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())
print('Agent:',tfl_agent.__file__)

In [ ]:
from google.colab import drive
from datetime import datetime, timezone
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/Colab Notebooks/agent/midterm demo')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')

In [ ]:
from tfl_agent.agent.phase6_common import extract_phase52, sha
ARCHIVE = BASE/'Phase52_Outputs'/'20261127T021948099654Z.zip'
if not ARCHIVE.is_file():
    other = ARCHIVE.with_name('20261127T021948099654Z (1).zip')
    if other.is_file(): ARCHIVE=other
    else: raise FileNotFoundError(f'Expected Phase52 run ZIP: {ARCHIVE}. Available: {[p.name for p in ARCHIVE.parent.glob("*.zip")]}')
CACHE=extract_phase52(ARCHIVE,Path('/content/Phase6_Cache')/RUN_ID)
print('Phase52 archive:',ARCHIVE, 'SHA256:',sha(ARCHIVE))

In [ ]:
from tfl_agent.agent.phase6_retrieval import run_retrieval
OUT=Path('/content/Phase62')/RUN_ID
report=run_retrieval(CACHE,OUT)
report['source_archive_sha256']=sha(ARCHIVE)
(OUT/'Summary.json').write_text(json.dumps(report,indent=2,ensure_ascii=False))
print(json.dumps(report,indent=2))
import pandas as pd
from IPython.display import display
display(pd.read_csv(OUT/'Known_Anchor_Check.csv'))
d=pd.read_csv(OUT/'Retrieval_Diagnostics.csv')
display(d.groupby('method')[['retrieved','historical_claim_overlap','different_topic_chunks']].mean())

In [ ]:
DEST=BASE/'Phase62_Outputs'
DEST.mkdir(parents=True,exist_ok=True)
saved=Path(shutil.make_archive(str(DEST/RUN_ID),'zip',root_dir=OUT))
print('Saved to Drive:',saved)